# 20260721_EDA_2016_전국_시도분리정제

- 작성자: 이정연
- #17 이슈 참고 

# 시행계획 시도 분리·정제 공통 유틸 사용 템플릿

이 파일을 복사한 뒤 파일명을 `YYYYMMDD_EDA_{연도}_전국_시도분리정제.ipynb` 형식으로 변경한다.

복사 후 반드시 확인할 항목:

1. `YEAR`, `SOURCE_FILE`, `SOURCE_SHEET`
2. 해당 연도의 0원 표기인 `ZERO_BUDGET_TOKENS`
3. 연도 고유 반복 머리글인 `EXTRA_HEADER_PATTERNS` — 2016~2020(제3차 기본계획) 원본은 시도별 단위표기 헤더 행("(단위 : 백만원)" 등)이 있는지 반드시 확인하고, 있으면 `UNIT_NOTATION_PATTERN`을 추가한다 (`src/features/pipeline_common.py` 참고, 이슈 #24)
4. **재원구분 이중계상 여부 `NEEDS_FUNDING_SOURCE_CLEANUP`** — 2016~2019(제3차 기본계획) 원본은 세부사업 하나가 계/국비/지방비(도비·시군비·비예산 등)로 최대 3~4행 중복 표기돼 있다. 원본 `사업분류재정구분` 값이 "계"/"국비"/"지방비" 등인지 확인 후 True로 변경한다 (`drop_exact_duplicate_rows`, `select_total_budget_rows`, 이슈 #26)
5. 소계 QA 허용오차인 `QA_TOLERANCE`
6. LLM 실행 여부와 연도별 체크포인트 경로

> 원본 예산 컬럼은 덮어쓰지 않는다. LLM 셀은 체크포인트와 API 키를 확인하기 전 실행하지 않는다.

In [8]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.llm_refine import (  # noqa: E402
    call_llm_once,
    clean_sentence,
    needs_llm_rerun,
    numbers_preserved,
)
from src.features.pipeline_common import (  # noqa: E402
    SUBTOTAL_LABEL_PATTERN,
    UNIT_NOTATION_PATTERN,
    assign_labels,
    build_subtotal_qa,
    calculate_budget_changes,
    classify_row,
    clean_text,
    drop_exact_duplicate_rows,
    get_sido_dir,
    normalize_budget_type,
    select_total_budget_rows,
    to_numeric_budget,
)
from src.features.text_match import (  # noqa: E402
    check_pattern,
    check_pattern_broad,
    dedup_label,
    extract_via_regex,
    find_near_duplicates,
)

# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시

print(f"프로젝트 루트: {PROJECT_ROOT}")

프로젝트 루트: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha


## 1. 연도·입력·예외 설정

아래 셀은 복사한 노트북에서 가장 먼저 수정한다. `비예산`, `·` 등을 0으로 처리할지는 해당 연도 원본을 확인한 뒤 결정한다.

In [9]:
YEAR = 2016
PREVIOUS_YEAR = YEAR - 1

CURRENT_BUDGET_COL = f"{YEAR}년 예산"
PREVIOUS_BUDGET_COL = f"{PREVIOUS_YEAR}년 예산"
CURRENT_NUM_COL = f"{YEAR}년_예산_num"
PREVIOUS_NUM_COL = f"{PREVIOUS_YEAR}년_예산_num"

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFIG_DIR = PROJECT_ROOT / "configs"

SOURCE_FILE = (
    DATA_DIR
    / "칼럼정렬"
    / "세부사업표추출_2016년도 지방자치단체 저출산고령사회 시행계획 (제3차 기본계획)_칼럼정렬.xlsx"
)
SOURCE_SHEET = "정리본_자동"
TABLE1_FILE = SOURCE_FILE  # Table 1 시트가 같은 파일 안에 있음

ZERO_BUDGET_TOKENS = ("-",)
# 2016은 시도별 "(단위 : 백만원)" 헤더 행(이슈 #24)과 총계/소계/합계 라벨 행(이슈 #29 예정)이 있음
EXTRA_HEADER_PATTERNS = (UNIT_NOTATION_PATTERN, SUBTOTAL_LABEL_PATTERN)
# 2016은 사업분류재정구분이 계/국비/지방비 등 재원 구분이라 이중계상 위험이 있음 (이슈 #26)
NEEDS_FUNDING_SOURCE_CLEANUP = True
QA_TOLERANCE = 0
RUN_LLM = False

CHECKPOINT_PATH = INTERIM_DIR / f"{YEAR}_llm_정제_체크포인트.csv"
QA_PATH = REPORTS_DIR / f"{YEAR}_전국_QA_검증결과.csv"

## 2. 데이터 로드와 기본 확인

In [10]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"SOURCE_FILE을 실제 파일로 수정하세요: {SOURCE_FILE}")

df_raw = pd.read_excel(SOURCE_FILE, sheet_name=SOURCE_SHEET)

required_columns = {
    "지역",
    "세부사업명",
    "사업분류재정구분",
    CURRENT_BUDGET_COL,
    PREVIOUS_BUDGET_COL,
    "주요내용",
    "원본행",
}
missing_columns = required_columns.difference(df_raw.columns)
if missing_columns:
    raise KeyError(f"입력 파일에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

print("데이터 크기:", df_raw.shape)
print("지역 수:", df_raw["지역"].nunique())
display(df_raw.head())

데이터 크기: (6882, 12)
지역 수: 17


,지역,세부사업명,사업분류재정구분,2016년 예산,2015년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,(단위 : 백만원),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,서울,총 계,NaN,4173069,4480127,-307058,-6.9,NaN,1.0,2.0,4.0,데이터행
2,서울,Ⅰ. 공통사업,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,5.0,구분행사업명 연속 후보
3,서울,공통사업 합계,계,3772641,4075595,-302954,-7.4,NaN,1.0,2.0,6.0,데이터행
4,서울,공통사업 합계,국비,1775591,2282149,-506558,-22.2,NaN,1.0,2.0,7.0,데이터행


## 3. 행 분류·계층 전파·숫자 변환

소계 QA에 원본 소계 행의 숫자값도 필요하므로 `df_labeled` 전체에 숫자 컬럼을 만든다.

In [11]:
if NEEDS_FUNDING_SOURCE_CLEANUP:
    df_raw = drop_exact_duplicate_rows(df_raw)
    df_raw = select_total_budget_rows(
        df_raw,
        budget_cols=[CURRENT_BUDGET_COL, PREVIOUS_BUDGET_COL],
        zero_tokens=ZERO_BUDGET_TOKENS,
    )
    print("재원구분(계/국비/지방비) 정리 후 행 수:", len(df_raw))

df_raw["사업행구분"] = df_raw["세부사업명"].apply(
    lambda value: classify_row(value, extra_header_patterns=EXTRA_HEADER_PATTERNS)
)

# TODO: 특정 연도에 예산이 모두 결측인 페이지 머리글이 있다면 여기서 헤더반복으로 보정

df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]
df_labeled[CURRENT_NUM_COL] = to_numeric_budget(
    df_labeled[CURRENT_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)
df_labeled[PREVIOUS_NUM_COL] = to_numeric_budget(
    df_labeled[PREVIOUS_BUDGET_COL], zero_tokens=ZERO_BUDGET_TOKENS
)

df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_labeled["사업행구분"].value_counts())
print("세부사업 행 수:", len(df_leaf))

display(df_labeled.head())

재원구분(계/국비/지방비) 정리 후 행 수: 4773
사업행구분
세부사업      4588
헤더반복       149
대분류_소계      36
Name: count, dtype: int64
세부사업 행 수: 4588


,세부사업명,사업분류재정구분,2016년 예산,2015년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역,2016년_예산_num,2015년_예산_num
141,(단위：백만원),(단위：백만원),(단위：백만원),(단위：백만원),(단위：백만원),(단위：백만원),(단위：백만원),144.0,3446.0,3463.0,데이터행,헤더반복,NaN,NaN,강원,<NA>,<NA>
142,총 계,NaN,966599,952834,13765,1.4,NaN,145.0,3464.0,3466.0,데이터행,헤더반복,NaN,NaN,강원,966599.0,952834.0
143,Ⅰ. 공통사업,NaN,NaN,NaN,NaN,NaN,NaN,145.0,3464.0,3467.0,구분행사업명 연속 후보,대분류_소계,Ⅰ. 공통사업,NaN,강원,<NA>,<NA>
752,공통사업 합계,계,836613,830867,5746,0.7,NaN,145.0,3464.0,3468.0,데이터행,헤더반복,Ⅰ. 공통사업,NaN,강원,836613.0,830867.0
144,1. 저출산 분야(공통사업),NaN,NaN,NaN,NaN,NaN,NaN,145.0,3464.0,3471.0,구분행사업명 연속 후보,세부사업,Ⅰ. 공통사업,NaN,강원,<NA>,<NA>


## 4. 텍스트 정제와 예산 재계산

In [13]:
for column in ["세부사업명", "대분류", "중분류"]:
    df_leaf[column] = clean_text(df_leaf[column])

df_leaf["주요내용"] = clean_text(df_leaf["주요내용"], strip_leading_bullet=True)
df_leaf["사업분류재정구분"] = normalize_budget_type(df_leaf["사업분류재정구분"])

budget_changes = calculate_budget_changes(
    df_leaf[CURRENT_NUM_COL],
    df_leaf[PREVIOUS_NUM_COL],
)
df_leaf[["당해예산", "전년도예산", "증감액", "증감율"]] = budget_changes
display(df_leaf[["지역", "세부사업명", "당해예산", "전년도예산", "증감액", "증감율"]].head(20))

,지역,세부사업명,당해예산,전년도예산,증감액,증감율
144,강원,1. 저출산 분야(공통사업),<NA>,<NA>,<NA>,<NA>
754,강원,보육비 지원확대,117814.0,123860.0,-6046.0,-4.9
755,강원,만3~5세 누리과정 확대,98756.0,109808.0,-11052.0,-10.1
756,강원,양육수당 지원 확대,43412.0,43186.0,226.0,0.5
757,강원,보육교지원 인건비 지원,60922.0,52648.0,8274.0,15.7
758,강원,공공형어린이집 지원,4600.0,4436.0,164.0,3.7
759,강원,어린이집 환경개선,930.0,928.0,2.0,0.2
760,강원,육아종합지원센터 운영 지원,548.0,380.0,168.0,44.2
761,강원,방과후과정 교육비 지원,12259.0,15471.0,-3212.0,-20.8
762,강원,입양가정 양육수당 지원,911.0,1131.0,-220.0,-19.5


## 5. 중분류 소계 QA와 저장

QA 계산은 공통 함수가 담당하고, 연도별 파일 저장은 노트북에서 담당한다.

In [14]:
qa = build_subtotal_qa(
    df_labeled,
    budget_col=CURRENT_NUM_COL,
    tolerance=QA_TOLERANCE,
)

display(qa["결과"].value_counts(dropna=False))
display(qa[qa["결과"] == "불일치"])

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
qa.to_csv(QA_PATH, index=False, encoding="utf-8-sig")
print(f"QA 저장 완료: {QA_PATH}")

결과
불일치    39
Name: count, dtype: int64

,지역,대분류,중분류,원본_소계값,leaf_합계,차이,오차율(%),QA_병합상태,결과,허용기준결과
0,강원,Ⅰ. 공통사업,NaN,<NA>,836343.0,<NA>,<NA>,leaf합계만,불일치,판정불가
1,강원,Ⅱ. 지자체 자체사업,NaN,<NA>,129987.0,<NA>,<NA>,leaf합계만,불일치,판정불가
2,경기,Ⅰ. 공통사업,NaN,<NA>,7894824.0,<NA>,<NA>,leaf합계만,불일치,판정불가
3,경기,Ⅱ. 지자체 자체사업,NaN,<NA>,450856.0,<NA>,<NA>,leaf합계만,불일치,판정불가
4,경기,NaN,NaN,<NA>,4938894.0,<NA>,<NA>,leaf합계만,불일치,판정불가
5,경남,Ⅰ. 공통사업,NaN,<NA>,1392401.0,<NA>,<NA>,leaf합계만,불일치,판정불가
6,경남,Ⅱ. 지자체 자체사업,NaN,<NA>,88783.0,<NA>,<NA>,leaf합계만,불일치,판정불가
7,경북,Ⅰ. 공통사업,NaN,<NA>,1214354.0,<NA>,<NA>,leaf합계만,불일치,판정불가
8,경북,Ⅱ. 지자체 자체사업,NaN,<NA>,444826.0,<NA>,<NA>,leaf합계만,불일치,판정불가
9,광주,NaN,NaN,<NA>,2559302.9,<NA>,<NA>,leaf합계만,불일치,판정불가


QA 저장 완료: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha/reports/2016_전국_QA_검증결과.csv


## 6. 주요내용 라벨 정리·부분 추출·유사 사업명 후보

In [15]:
df_leaf["주요내용"] = df_leaf["주요내용"].apply(dedup_label)
df_leaf["주요내용_패턴"] = df_leaf["주요내용"].apply(check_pattern)
df_leaf["주요내용_패턴_확장"] = df_leaf["주요내용"].apply(check_pattern_broad)

regex_extracted = pd.DataFrame(
    df_leaf["주요내용"].apply(extract_via_regex).tolist(),
    index=df_leaf.index,
)
df_leaf["지원대상"] = regex_extracted["지원대상"]
df_leaf["지원내용_상세"] = regex_extracted["지원내용"]

near_duplicates = find_near_duplicates(df_leaf, threshold=90)
print("유사 사업명 검토 후보:", len(near_duplicates))
display(near_duplicates.head(20))

유사 사업명 검토 후보: 144


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
0,전남,NaN,저소득층 기저귀조제분유 지원사업,20.0,"저소득층 영유아 가정 기저귀,조제분유 지원",저소득층 기저귀･조제분유 지원사업,26.0,저소득층 기저귀 및 분유 지원사업 - 중위소득 80% 이하인 자 - 출산 예정일 40일전부터 출산 후 30일 까지,97.142857
1,대전,NaN,산모 ․ 신생아 건강관리 지원,1831.0,"출산도우미 가정파견 서비스 제공 • 단태아(2주), 쌍태아(3주), 3태아이상(4주)",산모 ․ 신생아 건강관리사 지원,50.0,대상 : 중위소득 80%이하 • 지원 : 10일~20일,96.969697
2,경기,NaN,저출산･고령사회 대응기반 강화,1012.0,NaN,저출산고령사회 대응기반 강화,294.0,NaN,96.774194
3,경기,NaN,저출산･고령사회 대응기반 강화,1012.0,NaN,저출산고령사회 대응기반 강화,675.0,NaN,96.774194
4,전남,NaN,산모신생아 건강관리 지원,2442.0,"출산가정에 도우미서비스 지원을 통한 산모 및 신생아 건강관리 도모(4,140명)",산모신생아 건강관리사 지원,118.0,산모 200여명에게 산모신생아 2주간 돌봄서비스,96.296296
5,전남,NaN,산모신생아 건강관리 지원,54.0,출산가정에 도우미서비스지원을 통한 산모 및 신생아 건강관리 도모(41명),산모신생아 건강관리사 지원,118.0,산모 200여명에게 산모신생아 2주간 돌봄서비스,96.296296
6,전남,NaN,산모신생아 건강관리 지원,54.0,출산가정에 도우미서비스지원을 통한 산모 및 신생아 건강관리 도모(41명),산모신생아 건강관리사 지원,89.0,출산가정에 도우미 서비스지원을 통한 산모 및 신생아 건강관리 도모,96.296296
7,전남,NaN,산모신생아 건강관리 지원,2442.0,"출산가정에 도우미서비스 지원을 통한 산모 및 신생아 건강관리 도모(4,140명)",산모신생아 건강관리사 지원,89.0,출산가정에 도우미 서비스지원을 통한 산모 및 신생아 건강관리 도모,96.296296
8,충북,NaN,어린이집 아동간식 지원,175.0,학교 급식지원사업과 연계하여 어린이집 아동간식 지원,어린이집 아동간식비 지원,110.0,어린이집 아동간식비 지원,96.000000
9,충남,NaN,농어촌 방과후학교 지원,2709.0,"농어촌지역 방과후 학교 프로그램 운영비, 강사비, 재료비 등 지원",농산어촌 방과후학교 지원,200.0,농산어촌지역 44개교 지원,96.000000


## 7. 원본 Table 1 대조(필요 시)

원본 파일의 컬럼 배치가 다르면 `column_indices`를 해당 연도에 맞게 변경한다.

In [ ]:
# 예시: 실제 행 번호로 교체한 뒤 주석을 해제한다.
# df_table1 = pd.read_excel(TABLE1_FILE, sheet_name="Table 1", header=None)
# display(show_table1_around(df_table1, center_excel_row=100, window=3, label="원본 대조"))

## 8. LLM 보존형 교정(선택)

`RUN_LLM=True`로 바꾸기 전에 `.env`, 설정 파일, 연도별 체크포인트 파일명을 확인한다. 아래 셀은 연결 예시이며, 대량 실행 시 기존 노트북의 청크 단위 체크포인트 저장 로직을 함께 사용한다.

In [ ]:
if RUN_LLM:
    import os
    from functools import partial

    import yaml
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(PROJECT_ROOT / ".env")
    api_key = os.getenv("UPSTAGE_API_KEY")
    if not api_key:
        raise RuntimeError("UPSTAGE_API_KEY가 없습니다.")

    with (CONFIG_DIR / "llm_extraction.yaml").open(encoding="utf-8") as file:
        llm_cfg = yaml.safe_load(file)

    client = OpenAI(api_key=api_key, base_url=llm_cfg["upstage"]["base_url"])
    call_once = partial(call_llm_once, client=client, llm_config=llm_cfg)

    # 안전 확인용 단건 예시. 대량 처리는 체크포인트 로직을 붙인 뒤 수행한다.
    sample = df_leaf[df_leaf["주요내용"].notna()].iloc[0]
    sample_cleaned = clean_sentence(sample["세부사업명"], sample["주요내용"], call_once=call_once)
    print("원문:", sample["주요내용"])
    print("정제:", sample_cleaned)
    print("숫자보존:", numbers_preserved(sample["주요내용"], sample_cleaned))
    print("재실행대상:", needs_llm_rerun(sample_cleaned))
else:
    print("RUN_LLM=False: LLM 호출을 건너뜁니다.")

## 9. 시도별 최종 저장

최종 컬럼은 해당 연도 작업에서 확정한 스키마로 수정한다. 같은 파일을 중간과 최종 단계에서 중복 저장하지 않는다.

In [ ]:
output_cols = [
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]
missing_output_cols = set(output_cols).difference(df_leaf.columns)
if missing_output_cols:
    raise KeyError(f"최종 저장 전에 필요한 컬럼을 확인하세요: {sorted(missing_output_cols)}")

for sido, group in df_leaf.groupby("지역"):
    sido_dir = get_sido_dir(INTERIM_DIR, sido)
    output_path = sido_dir / f"{YEAR}_{sido}_세부사업_정제.csv"
    group[output_cols].to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"{sido}: {len(group)}행 저장 -> {output_path}")

## 완료 전 체크리스트

- [ ] 17개 시도 포함 여부 확인
- [ ] 행 유형 분포 확인
- [ ] 연도 고유 헤더·연속행 후보 원본 대조
- [ ] 재원구분(계/국비/지방비) 이중계상 여부 확인 및 `NEEDS_FUNDING_SOURCE_CLEANUP` 반영 (2016~2019)
- [ ] 중분류 소계 QA 불일치 원인 기록
- [ ] 증감액·증감율 재계산 결과 확인
- [ ] LLM 숫자 보존 불일치 원본 대조
- [ ] wide/long 행 수와 컬럼 스키마 확인
- [ ] 최종 CSV `utf-8-sig` 저장 확인
- [ ] 커널 재시작 후 전체 실행
- [ ] 리팩터링 전후 QA 수치 비교